## Lecture 16: Transportation problem
### 1. The LP Problem Formulation

**Sets**
    - $C = {1,2,3,4}$, the set of all cities
    - $P = {1,2,3}$, the set of all plants

**Decision Variables:**
Let $x_{ij}$ represent the amount of electricity (in million kWh) sent from Plant $i$ to City $j$, $\forall i \in P, j \in C$


**Objective Function:**
Minimize the total cost of transporting electricity:
$$\text{Minimize } Z = \sum_{i \in C, j \in P} c_{ij} x_{ij}$$

that is:
$$
\text{Minimize } Z = 8x_{11} + 6x_{12} + 10x_{13} + 9x_{14} + 9x_{21} + 12x_{22} + 13x_{23} + 7x_{24} + 14x_{31} + 9x_{32} + 16x_{33} + 5x_{34}
$$

**Constraints:**

1. Supply Constraints (for each plant):
    - $\sum x_{ij} = Supply_i, \forall i \in P$
    That is:
    - Plant 1: $x_{11} + x_{12} + x_{13} + x_{14} \leq 35$
    - Plant 2: $x_{21} + x_{22} + x_{23} + x_{24} \leq 50$ 
    - Plant 3: $x_{31} + x_{32} + x_{33} + x_{34} \leq 40$

2. Demand Constraints (for each city):
    - $\sum x_{ij} = demand_j \forall j \in CITIES$
    That is:
    - City 1: $x_{11} + x_{21} + x_{31} = 45$
    - City 2: $x_{12} + x_{22} + x_{32} = 20$
    - City 3: $x_{13} + x_{23} + x_{33} = 30$
    - City 4: $x_{14} + x_{24} + x_{34} = 30$

3. Non-Negativity Constraints:
   - $x_{ij} \geq 0; \quad \forall ( i, j )$

In [ ]:
# pip install gurobipy==11.0.0

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed March 20 16:52:37 2024
@author: npatankar
"""


# Code 2: Transportation Problem Lecture 16

import gurobipy as gp

m = 3 # Number of Supply Centers 
n = 4 # Number of Demand Centers 


# Step 2: Setup the parameters for the model
cost = [
    [8, 6, 10, 9],
    [9, 12, 13, 7],
    [14, 9, 16, 5]
]

supply = [35, 50, 40]           # Maximum supply
demand = [45, 20, 30, 30]       # Demand requirement

# Define the model
model = gp.Model("TransportationProblem")

# Decision variables
x = model.addVars(m, n, name="x")

# Objective function
model.setObjective(sum(cost[i][j] * x[i, j] for i in range(3) for j in range(4)), gp.GRB.MINIMIZE)

#Supply constraints
for i in range(m):
    model.addConstr(sum(x[i, j] for j in range(4)) <= supply[i], f"Supply_Plant_{i+1}")

# Demand constraints
for j in range(n):
    model.addConstr(sum(x[i, j] for i in range(3)) == demand[j], f"Demand_City_{j+1}")

# Solve the model
model.optimize()

In [ ]:

# Print the results
if model.status == gp.GRB.OPTIMAL:
    print("Optimal Solution Found:")
    Sol = []
    for var in model.getVars():
        Sol.append(var.x)
    x_sol = {(i+1,j+1): Sol[i*n+j] for i in range(m) for j in range(n)}

    ## Reduced Cost Calculations
    RC = []
    for var in model.getVars():
        RC.append(var.RC)
    x_RC =  {(i+1,j+1): RC[i*n+j] for i in range(m) for j in range(n)} # First N variables are x

    ## Calculation of Dual Variables
    pi = []
    for cons in model.getConstrs():
        pi.append(cons.pi)  
    Dual_Supply_Cons = {i+1: pi[i] for i in range(m)} # First m constraints are Supply Constraints
    Dual_Demand_Cons = {j+1: pi[m+j] for j in range(n)} # Next n constraints are Demand Constraints


    print(f"Total Cost = {model.ObjVal}")
    print(f"Value of x {x_sol}")
    print(f"RC of x {x_RC}")
    print(f"Dual of Supply Constraints {Dual_Supply_Cons}")
    print(f"Dual of Demand Constraints {Dual_Demand_Cons}")
else:
    print("No optimal solution found.")

In [ ]:
# Open a file to write the results
with open("Transportation_Model_Solution.txt", "w") as f:
    if model.status == gp.GRB.OPTIMAL:
        print(f"Total Cost = {model.ObjVal}")
        f.write(f"Optimal Objective Function Value, {model.ObjVal}\n")
        f.write(f"Value of x {x_sol}\n")
        f.write(f"RC of x {x_RC}\n")
        f.write(f"Dual of Supply Constraints {Dual_Supply_Cons}\n")
        f.write(f"Dual of Demand Constraints {Dual_Demand_Cons}\n")
    else:
        f.write("No optimal solution found.\n")